In [ ]:
# Install required libraries if not already installed
%pip install -q langchain langchain-community langchain-classic faiss-cpu pypdf wikipedia python-docx sentence-transformers

In [ ]:
# Imports and Setup
import os
import wikipedia
import textwrap
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from docx import Document

# LM Studio API (assume local server, e.g., http://localhost:1234)
import requests

# Memory for conversation
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)

print("[System] Libraries imported and memory initialized.")

In [ ]:
# Emmbeddings for vector storex
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# PDF Vector Store (initialized later if PDF found)
pdf_vectordb = None

def initialize_pdf_system():
    try:
        current_dir = os.getcwd() 
        pdf_files = [f for f in os.listdir(current_dir) if f.lower().endswith('.pdf')]
        
        if not pdf_files:
            return None

        target_pdf = os.path.join(current_dir, pdf_files[0])
        
        # --- The rest of your LangChain logic stays the same ---
        loader = PyPDFLoader(target_pdf)
        raw_docs = loader.load()
        
        splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
        docs = splitter.split_documents(raw_docs)
        
        return FAISS.from_documents(docs, embeddings)
        
    except Exception as e:
        print(f"[Error] Failed to index PDF: {e}")
        return None

print("[System] PDF system initialized.")

In [ ]:
# LM Studio Chatbot interaction
def lm_studio_chat(prompt, history=None, endpoint="http://localhost:1234/v1/chat/completions"):
    messages = []
    
    if history:
        for h in history:
            messages.append({"role": "user", "content": h[0]})
            messages.append({"role": "assistant", "content": h[1]})
    
    # always include current prompt
    messages.append({"role": "user", "content": prompt })
    
    payload = {
        "messages": messages,
        "model": "local-model",
        "stream": False
    }
    
    try:
        response = requests.post(endpoint, json=payload, timeout=120)
    except requests.RequestException as request_error:
        return f"[LM Studio Error] Request failed: {request_error}"
    
    try:
        data = response.json()
    except ValueError:
        return f"[LM Studio Error] Non-JSON response ({response.status_code}): {response.text}"
    
    if response.status_code >= 400:
        if isinstance(data, dict) and data.get("error"):
            return f"[LM Studio Error] {data['error']}"
        return f"[LM Studio Error] HTTP {response.status_code}: {data}"
    
    if isinstance(data, dict) and data.get("choices"):
        return data["choices"][0]["message"]["content"]
    
    if isinstance(data, dict) and data.get("error"):
        return f"[LM Studio Error] {data['error']}"
    
    return f"[LM Studio Error] Unexpected response: {data}"

print("[System] LM Studio chat function ready.")

In [ ]:
# Wiki search helper
def wiki_search(query):
    query = query.strip()
    if not query:
        return "No page found for that query."
    try:
        matches = wikipedia.search(query, results=5)
        if matches:
            page = wikipedia.page(matches[0], auto_suggest=False)
            return page.content  # full article text
        page = wikipedia.page(query, auto_suggest=True)
        return page.content  # full article text
    except wikipedia.exceptions.DisambiguationError as e:
        return f"Multiple results found: {e.options[:5]}"
    except wikipedia.exceptions.PageError:
        return "No page found for that query."
    except ValueError as value_error:
        return f"Wiki search error: {value_error}"
    except Exception as e:
        return f"Wiki search error: {e}"
    
print("[System] Wikipedia search function ready.")

In [ ]:
# Helper for docx creation
def create_docx(text, filename="output.docx"):
    doc = Document()
    doc.add_paragraph(text)
    doc.save(filename)
    return filename

print("[System] DOCX helper function ready.")

In [ ]:
# Print helper for long text
def print_wrapped(text, width=100):
    wrapper = textwrap.TextWrapper(width=width, replace_whitespace=False)
    lines = text.split('\n')
    wrapped_text = [wrapper.fill(line) for line in lines]
    print("\n".join(wrapped_text))

print("[System] Print helper function ready.")

In [ ]:
# Main chatbot loop

import re

conversation_history = []
conversation_stats = {
    "user_turns": 0,
    "wiki_searches": 0,
    "pdf_queries": 0,
    "arithmetic": 0,
    "ai_chats": 0
}

while True:
    q = input("You: [q to quit] ")
    print("\n[User] \n" + q)
    if q.strip().lower() == "q":
        try:
            lm_end = lm_studio_chat(
                """ Summarize the whole conversation history: """ + 
                str(conversation_history)
            )
        except Exception as summary_error:
            lm_end = f"Unable to generate conversation summary: {summary_error}"
        print("\n[LM Studio Conversation Summary]")
        print_wrapped(lm_end)
        print("\n--- Conversation Statistics ---")
        for k, v in conversation_stats.items():
            print(f"{k}: {v}")
        print("Goodbye!")
        break
    conversation_stats["user_turns"] += 1
    
    # LM Studio decides tool
    print("\n[LM Studio] Deciding which tool to use...")
    lm_response = lm_studio_chat(f"Analyze the user input and decide the tool to use: chat, wiki, arithmetic, pdf. Input: {q}. Output format: TOOL: <tool> | KEYWORDS: <keywords>")
    print(f"[LM Studio Tool Decision] {lm_response}")
    tool_match = re.search(r'TOOL: (\w+)', lm_response)
    tool = tool_match.group(1).lower() if tool_match else "chat"
    
    # Wiki search is a simple retrieval based on keywords extracted from LM response
    if tool == "wiki":
        print("[LM Studio] Using Wikipedia search tool...\n")
        conversation_stats["wiki_searches"] += 1
        keywords_match = re.search(r'KEYWORDS: (.+)', lm_response)
        keywords = keywords_match.group(1).splitlines()[0].strip() if keywords_match else q.strip()
        keywords = keywords.split(',')[0].strip() if keywords else q.strip()
        wiki_result = wiki_search(keywords)
        lm_response_wiki = lm_studio_chat(f"Summarize the following Wikipedia content in a concise manner. Input: {wiki_result} Output format: text")
        print("[Wiki]")
        print_wrapped(wiki_result)
        
        # Optional: LM Studio can format the wiki result into a docx file
        doc_choice = input("Create documentation in docx? (y/n): ")
        if doc_choice.lower().startswith("y"):
            lm_response_doc = lm_studio_chat(f"Format to documentation in docx style. Input: {wiki_result} Output format: text")
            safe_keyword = re.sub(r'[^A-Za-z0-9_-]+', '_', keywords).strip('_') or 'output'
            doc_filename = f"{safe_keyword}.docx"
            create_docx(lm_response_doc, filename=doc_filename)
            print(f"\nDocx created as {doc_filename}")
    
    # PDF analyzer is a vector search over the PDF content, then LM Studio analyzes the retrieved content to answer the question
    elif tool == "pdf":
        print("[LM Studio] Using PDF analysis tool...\n")
        conversation_stats["pdf_queries"] += 1
        
        if pdf_vectordb is None:
            print("[PDF Analyzer] Searching folder and indexing PDF...")
            pdf_vectordb = initialize_pdf_system()
        
        if pdf_vectordb is None:
            print("[PDF Analyzer] No PDF file found in folder or indexing failed.\n")
        else:
            search_results = pdf_vectordb.similarity_search(q, k=3)
            
            context = "\n\n".join([doc.page_content for doc in search_results])
            
            prompt = (
                f"""You are a helpful assistant analyzing a PDF document.
                Use the following context to answer the question.
                Context: {context}
                Question: {q}
                Answer based strictly on the context provided above. 
                Output format: text"""
            )
            
            lm_response_pdf = lm_studio_chat(prompt)
            
            print("[PDF Analyzer]")
            print_wrapped(lm_response_pdf)

    # Arithmetic Calc Solved by LM Studio reasoning through the problem
    elif tool == "arithmetic":
        print("[LM Studio] Using arithmetic calculation tool...\n")
        conversation_stats["arithmetic"] += 1
        
        lm_response_arith = lm_studio_chat(f"Evaluate the following arithmetic expression, reason about it and correctly answer: {q}. Output format: text")
        print(f"[Arithmetic] ")
        print_wrapped(lm_response_arith)
    
    # Default to LM Studio chat response
    else:
        print("[LM Studio] Ai is Thingking...\n")
        conversation_stats["ai_chats"] += 1
        answer = lm_studio_chat(q, history=conversation_history)
        print(f"[AI] ")
        print_wrapped(answer)
    
    # Update conversation history
    conversation_history.append((q, answer))
    if tool == "wiki":
        conversation_history.append((q, lm_response_wiki))
    if tool == "pdf":
        conversation_history.append((q, lm_response_pdf))
    if tool == "arithmetic":
        conversation_history.append((q, lm_response_arith))